In [2]:
#!pip install langchain-openai

# loading

In [3]:
import wikipedia

In [4]:
def load_document_from_wikipedia(title: str) -> str:
    """Fetches a Wikipedia page and returns its title and content as a single string."""
    try:
        page = wikipedia.page(title, auto_suggest=False)
        return f"{page.title}\n{page.content}"
        
    except (DisambiguationError, PageError) as e:
        raise ValueError(f"Could not retrieve page '{title}' from Wikipedia API.") from e


In [5]:
page = load_document_from_wikipedia("Porsche 911")
print(page[:500])

Porsche 911
The Porsche 911 model series (pronounced Nine Eleven or in German: Neunhundertelf, or colloquially Neunelfer) is a family of two-door, high performance rear-engine sports cars, introduced in September 1964 by Porsche of Stuttgart, Germany, and now in its eighth generation. All 911s have a rear-mounted flat-six engine, and usually 2+2 seating, except for special 2-seater variants. Originally, 911s had air-cooled engines, and torsion bar suspension, but the 911 has been continuously ch


# chunking

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
def chunk_document(
    document: str, 
    chunk_size: int = 1000, 
    chunk_overlap: int = 100) -> List[str]:
    """Splits a raw string document into smaller string chunks."""
    if not document.strip():
        return []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    return text_splitter.split_text(document)

In [8]:
chunks = chunk_document(page, chunk_size=1000, chunk_overlap=100)

In [24]:
len(chunks)

128

# embedding
- we follow rule of thumb to pass to openai embedding request:
  - document size (small): <500 chunks: pass all at once
  - document size (medium): 500 to 50000 chunks: pass 200 to 250 chunks at once
  - document size (large): >50000 chunks: 500 chunks + time.sleep (to avoid rate limit)

In [21]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
EMBEDDING_MODEL = "text-embedding-3-small"

In [23]:
openai_client = OpenAI()

In [29]:
from typing import List

def generate_embeddings(
    chunks: List[str], 
    model: str = EMBEDDING_MODEL, 
    batch_size: int = 250) -> List[List[float]]:
    """
    Generates cost-effective vector embeddings for a list of text chunks.
    Internally batches requests to prevent OpenAI Rate Limit and payload failures.
    """
    if not chunks:
        return []
        
    all_embeddings = []
    
    try:
        # Loop through the incoming texts in increments of batch_size
        for i in range(0, len(chunks), batch_size):
            batch_slice = chunks[i : i + batch_size]
            
            # Send the optimized slice to OpenAI
            print(f"Processing chunk #{i}")
            response = openai_client.embeddings.create(
                input=batch_slice,
                model=model
            )
            print("Processed!")
            print()
            print("---------------")
            print()
            
            # Extract and collect the embeddings for this specific batch
            batch_embeddings = [data.embedding for data in response.data]
            all_embeddings.extend(batch_embeddings)
            
        return all_embeddings
        
    except Exception as e:
        raise RuntimeError(f"OpenAI Embedding API operation failed: {str(e)}") from e


In [37]:
embeddings = generate_embeddings(chunks, batch_size=250)

Processing chunk #0
Processed!



# storing in vector database
- we will use chroma db (industry standard local vector database)
- always use embedding + vector storage inside one batching loop as industry standard
- always keep 500 to 1000 items per write operation as rule of thumb
- use generic identifier like doc_porsche_911_chunk_001 instead of uuid4 with the embedding 
- always use upsert (update + insert) to avoid duplication error

In [44]:
#!pip install chromadb

In [45]:
import os
from typing import List, Dict, Any
import chromadb
from openai import OpenAI

In [51]:
def ingest_documents_pipeline(
    collection_name: str,
    chunks: list[str],
    metadata_template: Dict[str, Any],
    batch_size: int = 250,
    embedding_model: str="text-embedding-3-small")-> chromadb.Collection:

    if not chunks:
        raise ValueError("The provided chunk list is completely empty.")
    
    openai_client = OpenAI()
    chroma_client = chromadb.PersistentClient(path="./chroma_db")
    collection = chroma_client.get_or_create_collection(name=collection_name)

    total_chunks = len(chunks)
    print(f"Executing unified ingestion pipeline for {total_chunks} chunks...")

    # Single loop for both embedding generation and storing in vectordb
    for i in range(0, total_chunks, batch_size):
        # 1. take the current batch 
        batch_chunks = chunks[i : i + batch_size]
        
        # 2. build readable ids and metadata for those chunks in batch
        batch_ids = [f"chunk_{j}" for j in range(i, i + len(batch_chunks))]
        batch_metadatas = [metadata_template for _ in range(len(batch_chunks))]

        # 3. convert the batch chunks into a vector matrix
        try:
            response = openai_client.embeddings.create(
                input=batch_chunks,
                model=embedding_model
            )
            batch_embeddings = [data.embedding for data in response.data]
        except Exception as api_error:
            raise RuntimeError(f"OpenAI API embedding generation failed at index offset {i}: {api_error}") from api_error

        # 4. add the batch assets to local storage using an upsert transaction
        try:
            collection.upsert(
                ids=batch_ids,
                embeddings=batch_embeddings,
                documents=batch_chunks,
                metadatas=batch_metadatas
            )
        except Exception as db_error:
            raise RuntimeError(f"Vector Database upload failed at index offset {i}: {db_error}") from db_error

        print(f"-> Successfully processed and saved Batch {i // batch_size + 1} (Chunks {i} to {i + len(batch_chunks) - 1})")
        
    print(f"\nPipeline Finished! Total active collection item count: {collection.count()}")
    return collection

In [54]:
metadata = {
    "source": "Wikipedia API",
    "topic": "Porsche 911",
    "data_format": "raw_string"
}

In [55]:
vector_db_collection = ingest_documents_pipeline(
    collection_name="porsche_knowledge_base",
    chunks=chunks,
    metadata_template=metadata,
    batch_size=250,
    embedding_model="text-embedding-3-small"
)

Executing unified ingestion pipeline for 128 chunks...
-> Successfully processed and saved Batch 1 (Chunks 0 to 127)

Pipeline Finished! Total active collection item count: 128


# inspect the vector db

In [56]:
import chromadb

# 1. Connect to the existing local directory
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# 2. Grab the collection we created earlier
collection = chroma_client.get_collection(name="porsche_knowledge_base")

print("--- Vector DB Health Check ---")
print(f"Total records in database: {collection.count()}")

# 3. Peek at the first 2 items (returns ids, documents, metadatas, and embeddings)
sample_data = collection.peek(limit=2)

print("\n--- Sample Item ID ---")
print(sample_data["ids"][0])

print("\n--- Sample Text Chunk ---")
print(sample_data["documents"][0][:200] + "...")

print("\n--- Sample Vector Embedding Dimensions ---")
# Check the length of the floating-point list (should be 1536 for text-embedding-3-small)
print(f"Vector Dimensions: {len(sample_data['embeddings'][0])}")


--- Vector DB Health Check ---
Total records in database: 128

--- Sample Item ID ---
chunk_0

--- Sample Text Chunk ---
Porsche 911
The Porsche 911 model series (pronounced Nine Eleven or in German: Neunhundertelf, or colloquially Neunelfer) is a family of two-door, high performance rear-engine sports cars, introduced ...

--- Sample Vector Embedding Dimensions ---
Vector Dimensions: 1536


# Data ingestion pipeline finished!